## **Imports**

In [9]:
from dotenv import load_dotenv
import os
import json

from langchain_openai import ChatOpenAI


In [11]:
# Change from langchain.memory -> langchain_classic.memory
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain


In [12]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

print("API Key Loaded:", bool(api_key))

API Key Loaded: True


## **Initialize LLM**

In [13]:
llm = ChatOpenAI(
    model="meta-llama/llama-3-8b-instruct",
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    temperature=0.3
)

print("LLM Ready")

LLM Ready


## **Add Memory**

In [14]:
memory = ConversationBufferMemory()

print("Memory initialized")

Memory initialized


C:\Users\ShoaibMohdKhan\AppData\Local\Temp\ipykernel_2768\1040275777.py:1: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory()


## **Create Conversation Chain**

In [15]:
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

print("Conversation chain ready")

Conversation chain ready


C:\Users\ShoaibMohdKhan\AppData\Local\Temp\ipykernel_2768\2779574555.py:1: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build a conversational agent with `langchain.agents.create_agent` and persist message history via a LangGraph checkpointer.
  conversation = ConversationChain(


## **Reservation System Prompt**

In [16]:
system_prompt = """
You are a restaurant reservation assistant.

Your responsibilities:
- Collect reservation details
- Ask follow-up questions for missing information
- Be polite and professional

You must collect:
1. Customer name
2. Reservation date
3. Reservation time
4. Number of guests

Do not finish the conversation until all details are collected.
"""

## **Start Conversation**

In [17]:
user_message = """
I want to reserve a table tomorrow evening
"""

full_prompt = system_prompt + "\nUser: " + user_message

response = conversation.predict(input=full_prompt)

print(response)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: 
You are a restaurant reservation assistant.

Your responsibilities:
- Collect reservation details
- Ask follow-up questions for missing information
- Be polite and professional

You must collect:
1. Customer name
2. Reservation date
3. Reservation time
4. Number of guests

Do not finish the conversation until all details are collected.

User: 
I want to reserve a table tomorrow evening

AI:

> Finished chain.
Welcome to our restaurant! I'd be happy to help you with your reservation. To confirm, you'd like to make a reservation for tomorrow evening, correct? Can you please tell me your name, so I can associate it with your reservation?


In [18]:
response = conversation.predict(
    input="Reservation is under Shoaib for 4 people at 8 PM"
)

print(response)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: 
You are a restaurant reservation assistant.

Your responsibilities:
- Collect reservation details
- Ask follow-up questions for missing information
- Be polite and professional

You must collect:
1. Customer name
2. Reservation date
3. Reservation time
4. Number of guests

Do not finish the conversation until all details are collected.

User: 
I want to reserve a table tomorrow evening

AI: Welcome to our restaurant! I'd be happy to help you with your reservation. To confirm, you'd like to make a reservation for tomorrow evening, correct? Can you please tell me your name, so I can associate it with your reservation?
Human: Reservation is under Shoaib for 4 peop

## **Structured Reservation Extraction**

In [22]:
extract_prompt = """
Extract reservation details from the conversation.

Return ONLY raw JSON.
Do not add explanations.
Do not use markdown.
Do not use ```json.

Conversation:
User wants reservation tomorrow evening.
Reservation under Shoaib for 4 people at 8 PM.

Required format:
{
    "name": "",
    "date": "",
    "time": "",
    "guests": 0
}
"""

response = llm.invoke(extract_prompt)

print(response.content)

{
    "name": "Shoaib",
    "date": "tomorrow",
    "time": "8 PM",
    "guests": 4
}


## **Convert to Python Dictionary**

In [24]:
cleaned_response = response.content.strip()

# Remove markdown if present
cleaned_response = cleaned_response.replace("```json", "")
cleaned_response = cleaned_response.replace("```", "")

reservation_data = json.loads(cleaned_response)

print(type(reservation_data))
print(reservation_data)

<class 'dict'>
{'name': 'Shoaib', 'date': 'tomorrow', 'time': '8 PM', 'guests': 4}
